# 01 - Explore

Scaffold only - no pre-written analysis. Load data, then work below.

In [ ]:
import sys

sys.path.insert(0, "..")  # repo root, so `from src import ...` resolves regardless of the kernel's cwd

import pandas as pd

from src import clean, confounder, margin, schema
from src import slice as slice_


## Load data

In [ ]:
ledger = pd.read_csv("../data/processed/ledger.csv", parse_dates=[schema.TXN_DATE, schema.PAYMENT_DATE])
overhead = pd.read_csv("../data/processed/overhead.csv", parse_dates=[schema.PERIOD_MONTH])


## Compute margin

In [ ]:
ledger_with_margin = margin.compute_margin(ledger, overhead, "proportional_by_revenue")
ledger_with_margin[[schema.TXN_ID, schema.REVENUE, margin.ALLOCATED_COST, margin.MARGIN, margin.MARGIN_PCT]].head()


## Slice

## Check confounders

Each helper below returns a comparison table, not a verdict. The entities/periods/dimension picked here (top two customers by revenue, the first two months, `origin_id`) are arbitrary choices to exercise the code paths against the synthetic (patternless) dataset - swap in whatever specific comparison you actually want to check once real data replaces it.

In [ ]:
from IPython.display import display

ledger_with_margin["_period"] = margin.period_month(ledger_with_margin[schema.TXN_DATE])
periods = sorted(ledger_with_margin["_period"].dropna().unique())
customers_by_revenue = ledger_with_margin.groupby(schema.CUSTOMER_ID)[schema.REVENUE].sum().sort_values(ascending=False)
top_two_customers = customers_by_revenue.index[:2].tolist()

print("stratified_comparison: top two customers, held fixed by origin_id")
display(confounder.stratified_comparison(
    ledger_with_margin,
    fixed_col=schema.ORIGIN_ID,
    compare_col=schema.CUSTOMER_ID,
    group_a=top_two_customers[0],
    group_b=top_two_customers[1],
))

print("period_over_period_excluding: first two months, excluding the top customer")
display(confounder.period_over_period_excluding(
    ledger_with_margin,
    entity_col=schema.CUSTOMER_ID,
    entity_value=customers_by_revenue.index[0],
    period_a=periods[0],
    period_b=periods[1],
    period_col="_period",
))

print("mix_shift_decomposition: same two months, by origin_id")
display(confounder.mix_shift_decomposition(
    ledger_with_margin,
    period_col="_period",
    period_a=periods[0],
    period_b=periods[1],
    group_col=schema.ORIGIN_ID,
))

print("accrual_vs_cash_view: revenue booked by txn_date vs by payment_date")
display(confounder.accrual_vs_cash_view(ledger_with_margin))
